<a href="https://colab.research.google.com/github/harley1983/final-exam-QuynhDinh/blob/main/section3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 3.1 - Issue Analysis with Course Connection


Here are the main problems with the current `safe_weather_data_fetch()` function:

1. **Overly broad exception handling**  
   - **Week 11: Files & Exception Handling**  
   The use of `except:` without specifying an exception type catches *all* errors, including critical ones like `KeyboardInterrupt` and `SystemExit`. This makes debugging harder and hides real problems.

2. **Non-descriptive error return**  
   - **Week 12: Testing and Debugging**  
   Returning `"Error occurred"` as a string gives no clear signal to the calling function. It’s better to return `None`, or a dictionary that indicates failure, so it can be programmatically checked.

3. **Silent failures with no error context**  
   - **Week 8 & Week 11: JSON parsing and exception handling**  
   If the city is invalid or the JSON is malformed, the function fails silently. Users get no information about *why* it failed, which violates good user feedback practices.

4. **No input validation**  
   - **Week 2 & Week 4: Input checking fundamentals**  
   The function doesn’t check if the `city` parameter is empty or invalid before making the API call, which could waste network resources or cause avoidable errors.

5. **Assumes API response structure**  
   - **Week 6 & Week 8: Defensive programming and nested structures**  
   The function directly accesses keys like `['current_condition'][0]['temp_C']` without checking if those keys exist, risking `KeyError` or `IndexError` if the API changes or is incomplete.


**Prompts AI I used to refine my code**:
- Prompt 1: I need to simplify the error reporting in this function. Collapse the API request and JSON parsing into a single try/except block. Use print() for errors
- Prompt 2: Help me write doctest examples for the function. I need tests that show: A blank input, An invalid city (SKIPPED), A successful call to "London" (SKIPPED)

In [ ]:
import requests

def refined_safe_weather_data_fetch(city):
    """
    Fetch weather data with basic error handling

    >>> refined_safe_weather_data_fetch("")
    Error: City name cannot be empty
    >>> refined_safe_weather_data_fetch("InvalidCity123")  # doctest: +SKIP
    Error: Could not connect to weather service
    >>> refined_safe_weather_data_fetch("London")  # doctest: +SKIP
    {'city': 'London', 'temperature': '15', 'wind_speed': '10', 'description': 'Partly cloudy'}
    """


    if city == "":
        print("Error: City name cannot be empty")
        return None

    if city == None:
        print("Error: City name cannot be empty")
        return None

    # Check for whitespace-only input
    if city.strip() == "":
        print("Error: City name cannot be empty")
        return None

    # Try to connect to weather service and get data
    try:
        url = "http://wttr.in/" + city + "?format=j1"
        response = requests.get(url)
        data = response.json()
    except:
        print("Error: Could not connect to weather service")
        return None

    # Try to extract weather information from response
    try:
        current = data["current_condition"]
        weather = current[0]
        temp = weather["temp_C"]
        wind = weather["windspeedKmph"]
        desc_list = weather["weatherDesc"]
        desc_dict = desc_list[0]
        description = desc_dict["value"]

        result = {
            "city": city,
            "temperature": temp,
            "wind_speed": wind,
            "description": description
        }
        return result
    except:
        print("Error: Could not read weather data")
        return None

#  Test function
def test_the_function():
    """Test function with basic examples"""

    print("Testing the weather function:")
    print("")

    # Test empty string
    print("Testing empty string:")
    result1 = refined_safe_weather_data_fetch("")
    print("Result:", result1)
    print("")

    # Test with spaces
    print("Testing with spaces:")
    result2 = refined_safe_weather_data_fetch("   ")
    print("Result:", result2)
    print("")

    # Test valid city
    print("Testing London:")
    result3 = refined_safe_weather_data_fetch("London")
    print("Result:", result3)
    print("")

    # Test invalid city
    print("Testing invalid city:")
    result4 = refined_safe_weather_data_fetch("InvalidCity123")
    print("Result:", result4)

if __name__ == "__main__":
    # Run the doctests
    import doctest
    doctest.testmod()

    # Run basic tests
    test_the_function()

### 🔍 3.3 Comparison Analysis – Refined vs. Original `safe_weather_data_fetch()`

#### 2 Similarities
1. **Use of try/except blocks:**  
   Both versions wrap external API calls and data extraction in `try/except` blocks to prevent program crashes from unexpected errors.

2. **Same core functionality:**  
   Both aim to retrieve real-time weather data from `wttr.in`, parse the JSON response, and return a dictionary with keys: `'city'`, `'temperature'`, `'wind_speed'`, and `'description'`.

---

#### 2 Differences
1. **Stronger input validation in refined version:**  
   The new version checks for both empty strings and strings with only spaces (`city.strip() == ""`), making it more user-proof. The original version only checks `if not city`.

2. **Simplified error structure to follow Week 8 constraints:**  
   The refined version groups `requests.get()` and `.json()` parsing into a single `try` block. This reduces complexity and meets the requirement to use only one `except` block per operation, unlike the original which used separate blocks.

---

#### Course Connection
This refined function reflects concepts from **Week 8 – APIs and External Data**.  
It demonstrates:  
- Manual URL creation with string concatenation  
- Basic input validation  
- Simple `try/except` structure without advanced exception handling  
- Use of JSON and dictionaries in their most basic form

---

#### 1 Improvement Area (Based on Textbook Chapter 6: Defensive Programming)
While the function uses `try/except` to catch missing keys, a better practice (as described in Chapter 6) would be to **explicitly check if keys like `'current_condition'` and `'weatherDesc'` exist** before accessing them. This would reduce reliance on catching errors and improve clarity and maintainability.

